# Notebook de construction des jeux de données à partir du scraping de l'API OpenAlex

In [ ]:
# modules 
import pandas as pd
from collections import Counter
from itertools import chain

In [ ]:
df = pd.read_csv("C:\Users\sacha\Downloads\openalex_vietnam_fr_us_ready_ter.csv") #résultats du scraping

## 1. Filtrage de données du scraping

Filtrage sur le type de publications

In [71]:
df["type"].value_counts()

type
article            108760
preprint             6478
review               4861
book-chapter         4773
dissertation         1656
book                 1327
report               1226
paratext              712
letter                630
editorial             390
dataset               266
peer-review           175
other                 146
erratum                85
reference-entry        13
retraction              6
Name: count, dtype: int64

In [72]:
df = df[~df["type"].isin(["paratext", "other", "editorial", "letter", "reference-entry", "erratum", "peer-review", "dataset"])].reset_index(drop=True)

Restriction aux publications de sciences sociales

In [73]:
df = df[df["primary_topic_domain_name"].notna()].reset_index(drop=True) # le filtrage repose sur cette colonne donc nous enlevons d'abord les lignes à valeurs manquantes
df = df[df["domains"].notna()] # de même, pour le second filtrage, qui se veut plus fin

In [76]:
df["primary_topic_domain_name"].value_counts() # niveau de granularité très bas qui permet un premier filtrage efficace

primary_topic_domain_name
Social Sciences    50849
Name: count, dtype: int64

In [77]:
df = df[df["primary_topic_domain_name"].isin(["Social Sciences"])].reset_index(drop=True)

In [78]:
df["domains"] # niveau de granularité plus fin, et plusieurs valeurs possibles

0        Business; Economics; Medicine; Mathematics; Ph...
1                        Mathematics; Economics; Sociology
2        Political science; Economics; Mathematics; Phi...
3        Geography; Political science; Economics; Socio...
4                  Medicine; Biology; Economics; Sociology
                               ...                        
50844                                      Philosophy; Art
50845                                                  Art
50846                                         Art; History
50847                                              History
50848                         Political science; Sociology
Name: domains, Length: 50849, dtype: object

In [79]:
count = Counter(chain.from_iterable(v.split('; ') for v in df["domains"].dropna())) # permet de compter les occurences dans une colonne avec des valeurs multiples pour chaque ligne (séparées par des "; ")
count

Counter({'Political science': 26618,
         'Economics': 18215,
         'Sociology': 17287,
         'Philosophy': 16194,
         'Business': 15601,
         'Psychology': 15371,
         'Computer science': 14845,
         'Geography': 9328,
         'Medicine': 9321,
         'History': 8313,
         'Biology': 6971,
         'Mathematics': 5968,
         'Art': 5794,
         'Engineering': 5273,
         'Physics': 4362,
         'Chemistry': 2553,
         'Environmental science': 1251,
         'Geology': 621,
         'Materials science': 250})

Filtrage plus fin sur les domaines 

In [80]:
df = df[~((df["domains"].str.contains("Medicine")) | (df["domains"].str.contains("Biology")) | (df["domains"].str.contains("Geology")) | (df["domains"].str.contains("Physics")) | (df["domains"].str.contains("Computer science")) | (df["domains"].str.contains("Chemistry")) | (df["domains"].str.contains("Materials science")) | (df["domains"].str.contains("Engineering")) | (df["domains"].str.contains("Environmental science"))| (df["domains"].str.contains("Mathematics")))].reset_index(drop=True)

In [81]:
df

,id,doi,title,publication_year,publication_date,cited_by_count,language,type,institution_countries,institutions,...,topics,primary_topic_id,primary_topic_name,primary_topic_field_name,primary_topic_subfield_name,primary_topic_domain_name,primary_source_name,primary_source_type,referenced_ids,referenced_count
0,https://openalex.org/W3024878631,https://doi.org/10.1016/j.jdeveco.2010.07.004,The long-run impact of bombing Vietnam,2010,2010-07-24,591,en,article,GB; US,Centre for Economic Policy Research; Universit...,...,Strategic bombing,https://openalex.org/T12825,"Defense, Military, and Policy Studies","Economics, Econometrics and Finance",Economics and Econometrics,Social Sciences,Journal of Development Economics,journal,https://openalex.org/W2171656257; https://open...,79
1,https://openalex.org/W2133466515,https://doi.org/10.1001/archpsyc.1987.01800230...,Psychophysiologic Assessment of Posttraumatic ...,1987,1987-11-01,764,en,article,US,United States Department of Veterans Affairs,...,NaN,https://openalex.org/T10242,Posttraumatic Stress Disorder Research,Psychology,Clinical Psychology,Social Sciences,Archives of General Psychiatry,journal,https://openalex.org/W1971449933; https://open...,18
2,https://openalex.org/W3123011427,https://doi.org/10.1162/003355399556278,Interfirm Relationships and Informal Credit in...,1999,1999-11-01,930,en,article,US,"Stanford University; University of California,...",...,NaN,https://openalex.org/T12544,Working Capital and Financial Performance,"Business, Management and Accounting",Accounting,Social Sciences,The Quarterly Journal of Economics,journal,https://openalex.org/W3123550268; https://open...,32
3,https://openalex.org/W2041685217,https://doi.org/10.1017/s0020818300028198,Learning and foreign policy: sweeping a concep...,1994,1994-01-01,740,en,article,US,"Rutgers, The State University of New Jersey",...,Foreign policy,https://openalex.org/T10053,International Relations and Foreign Policy,Social Sciences,Political Science and International Relations,Social Sciences,International Organization,journal,https://openalex.org/W5855877; https://openale...,140
4,https://openalex.org/W2049228636,https://doi.org/10.2307/2600916,The Pretty Prudent Public: Post Post-Vietnam A...,1992,1992-03-01,546,en,article,US,"University of California, Davis",...,Public opinion,https://openalex.org/T14086,Military and Defense Studies,Social Sciences,Political Science and International Relations,Social Sciences,International Studies Quarterly,journal,https://openalex.org/W183747230; https://opena...,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17699,https://openalex.org/W3035061665,https://doi.org/10.35537/10915/96082,Historia Social Contemporánea,2020,2020-05-01,0,es,book,AR; US,Universidad Nacional de La Plata; Weatherford ...,...,NaN,https://openalex.org/T12493,"Memory, violence, and history",Psychology,Social Psychology,Social Sciences,Papel Cosido eBooks,ebook platform,https://openalex.org/W6846242187; https://open...,26
17700,https://openalex.org/W4312999998,https://doi.org/10.31338/uw.9788323557197.pp.2...,Georges Didi-Huberman : penser par les insects,2022,2022-01-01,0,fr,book-chapter,BR; GB; PL; US,Adam Mickiewicz University in Poznań; Centro U...,...,NaN,https://openalex.org/T11132,Psychoanalysis and Psychopathology Research,Psychology,Clinical Psychology,Social Sciences,NaN,NaN,https://openalex.org/W6751658746,1
17701,https://openalex.org/W4312622443,https://doi.org/10.31338/uw.9788323557197.pp.4...,Secouer le lecteur. Perspective narrative et f...,2022,2022-01-01,0,fr,book-chapter,BR; GB; PL; US,Adam Mickiewicz University in Poznań; Centro U...,...,NaN,https://openalex.org/T12461,Narrative Theory and Analysis,Arts and Humanities,Literature and Literary Theory,Social Sciences,NaN,NaN,https://openalex.org/W2038655596; https://open...,7
17702,https://openalex.org/W2809570489,NaN,The writings of Monica Hughes : implications f...,1987,1987-01-01,0,en,article,US,The 

Dropping des colonnes inutiles 

In [82]:
df = df.drop(columns=["doi", "publication_date", "cited_by_count", "abstract", "primary_topic_id", 'primary_source_name', 'primary_source_type'])

drop duplicat dans les titres (doi ?)

In [83]:
df= df.drop_duplicates(subset=["title"]).reset_index(drop=True)

drop valeurs manquantes pour les colonnes d'intérêt

In [85]:
df = df[df["primary_topic_name"].notna()]
df = df[df["subfields"].notna()].reset_index(drop=True)

## 2. Création de colonnes

La mal nommée colonne "lang_type"

In [87]:
# boucle permettabt de ne garder que l'information sur les collaborateurs français, américains ou vietnamiens et de la résumer

sub1 = {"US"}
sub2 = {"FR"}
sub3 = {"VN"}
sub4 = {"US", "FR", "VN"}
lang_type = []
for i in df["institution_countries"]:
    j = i.split("; ")
    if sub4.issubset(set(j)) : 
        lang_type.append("franco_américano_vietnamien")
    elif sub1.issubset(j) and sub2.issubset(j) and not sub3.issubset(j) :
        lang_type.append("franco_américain")
    elif sub1.issubset(j) and sub3.issubset(j) and not sub2.issubset(j) :
        lang_type.append("américano-vietnamien")
    elif sub2.issubset(j) and sub3.issubset(j) and not sub1.issubset(j) :
        lang_type.append("franco-vietnamien")
    elif sub1.issubset(j) and not sub2.issubset(j) and not sub3.issubset(j):
        lang_type.append("américain")
    elif sub2.issubset(j) and not sub1.issubset(j) and not sub3.issubset(j) :
        lang_type.append("français")
    else:
        lang_type.append("vietnamien")
df["lang_type"] = lang_type

colonne références citées internes au jeu de données

In [88]:
# identifiant (normalement) uniques des articles
id_set = set(df["id"])

# expansion, filtrage puis regroupement un peu comme sur openrefine)  de la colonne referenced_ids (liste de tous les articles cités)
df_exploded = (df.assign(ref_list=df["referenced_ids"].str.split("; ")).explode("ref_list"))
# articles cités en interne
df_filtered = df_exploded[df_exploded["ref_list"].isin(id_set)]
df_internal_refs = (df_filtered.groupby(df_filtered.index)["ref_list"].apply(list).reindex(df.index, fill_value=pd.NA))

In [89]:
df["referenced_ids_internal"] = df_internal_refs

In [ ]:
#df.to_csv("openalex_vietnam_new_year_to_use.csv.csv", index=False, encoding="utf-8")

## 3. Version "réseau de citations"

version avec reference ids

In [ ]:
# On ne garde que les publications pour lesquelles la colonne des références citées en interne n'est pas vide
d = df[df["referenced_ids_internal"].notna()].reset_index(drop=True)

In [ ]:
#d.to_csv("C:\Users\sacha\Downloads\openalex_vietnam_new_year_family.csv", index=False, encoding="utf-8")